In [30]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
import time
import os
import pandas as pd

In [31]:
def set_up_driver():
    chrome_options = Options()
    #chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [32]:
def get_medicine_menu_links(driver):
    # Lấy tất cả các liên kết danh mục thuốc
    menu = []
    try:
         menu_items = driver.find_elements(By.XPATH, "//*[@id='pmc-v2']/div[4]/div/div[2]/div[1]//a")
    except TimeoutException:
        print("Không tìm thấy các liên kết danh mục thuốc.")
        return []

    for item in menu_items:
            # link = item.get_attribute("href")
            # menu_links.append(link)
            menu.append({
            "drug_type": item.text.strip(),
            "menu_link": item.get_attribute("href")
            })
    return menu

In [33]:
# def get_medicine_category_links(driver, menu_link):
#     category_links = []
#     try:
#         driver.get(menu_link)
#         time.sleep(5)
#         category_elements = driver.find_elements(By.XPATH, "//*[@id='pmc-v2']/div[4]/div/div[2]/div[1]//a")
#         for element in category_elements:
#             link = element.get_attribute("href")
#             category_links.append(link)
#     except TimeoutException:
#         print("Không tìm thấy các liên kết danh mục thuốc.")
#     return category_links

In [34]:
def get_medicine_category_links(driver, menu_link):
    driver.get(menu_link)
    time.sleep(3)
    category_elements = driver.find_elements(By.XPATH, "//*[@id='pmc-v2']/div[4]/div/div[2]/div[1]//a")
    categories = []
    for el in category_elements:
        categories.append({
            "category_name": el.text.strip(),
            "category_link": el.get_attribute("href")
        })
    return categories

In [35]:
# def get_medicine_links(driver,category_link):
#     medicine_links = []
#     try:
#         driver.get(category_link)
#         time.sleep(5)
#         medicine_elements = driver.find_elements(By.XPATH,"//div[contains(@class,'product-list')]//div[contains(@class,'product-card-new')]//a[@href]")
#         for element in medicine_elements:
#             link = element.get_attribute("href")
#             medicine_links.append(link)
#     except TimeoutException:
#         print("Không tìm thấy các liên kết thuốc.")
#     finally:
#         driver.quit()
#     return medicine_links

In [36]:
def get_medicine_links(driver, category_link):
    medicine_links = []
    try:
        driver.get(category_link)
        wait = WebDriverWait(driver, 10)
        time.sleep(3)

        # TÌm và bấm nút "Xem thêm" đến khi hết
        while True:
            try:
                btn = wait.until(EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[contains(., 'Xem thêm') or contains(., 'Xem Thêm')]"
                )))
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                time.sleep(1)
                btn.click()
                time.sleep(2)  
            except Exception:
                break  

        medicine_elements = driver.find_elements(
            By.XPATH, "//div[contains(@class,'product-list')]//div[contains(@class,'product-card-new')]//a[@href]"
        )
        for element in medicine_elements:
            medicine_links.append(element.get_attribute("href"))

    except TimeoutException:
        print("Không tìm thấy các liên kết thuốc.")

    return list(dict.fromkeys(medicine_links))

In [37]:
def crawl_pharmacity_medicine_links():
    driver = set_up_driver()
    wait = WebDriverWait(driver, 10)
    records = []
    try: 
        print("Đang truy cập vào trang chủ Pharmacity...")
        driver.get("https://www.pharmacity.vn/")
        time.sleep(10)


        # pop_up = driver.find_element(By.CLASS_NAME, "flex w-auto items-center justify-center")
        # time.sleep(2)
        # close_pop_up = pop_up.find_element(By.XPATH,"//*[@id='radix-:ra2:']/div[2]/div/button")
        # close_pop_up.click()
        # time.sleep(2)

        print("Đang truy cập vào trang danh mục thuốc...")
        thuoc_page = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#pmc-top-category > div.hidden.items-center.gap-3.text-white.md\:flex.xl\:gap-6 > a:nth-child(1) > span > span")))
        thuoc_page.click()
        time.sleep(3)

        menu = get_medicine_menu_links(driver)
        # print(f"Số danh mục thuốc thu thập được: {len(menu_links)}")
        # for link in menu_links:
        #     print(link)
       
        
        # print("Thu thập danh sách danh mục thuốc")
        # all_category_links = []
        # for menu_link in menu_links:
        #     category_links = get_medicine_category_links(driver, menu_link)
        #     all_category_links.extend(category_links)
        # print(f"Tổng số liên kết danh mục thuốc thu thập được: {len(all_category_links)}")

        # for link in all_category_links:
        #     print(link)

        for i, m in enumerate(menu, 1):
            print(f"[{i}/{len(menu)}] Menu: {m['drug_type']}")
            categories = get_medicine_category_links(driver, m["menu_link"])
            for j, c in enumerate(categories, 1):
                print(f"  [{j}/{len(categories)}] Category: {c['category_name']}")
                links = get_medicine_links(driver, c["category_link"])
                print(f"    -> Links: {len(links)}")
                for link in links:
                    print(f"      {link}")
                    records.append({
                        "links": link,
                        "drug_type": m["drug_type"],
                        "category_link": c["category_link"],
                        "category_name": c["category_name"]
                    })
    
    except Exception as e:
        print(f"Lỗi: {e}")
        return []

    finally:
        driver.quit()
    return pd.DataFrame(records)

<>:18: SyntaxWarning: invalid escape sequence '\:'
<>:18: SyntaxWarning: invalid escape sequence '\:'
C:\Users\Admin\AppData\Local\Temp\ipykernel_28616\3786770194.py:18: SyntaxWarning: invalid escape sequence '\:'
  thuoc_page = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#pmc-top-category > div.hidden.items-center.gap-3.text-white.md\:flex.xl\:gap-6 > a:nth-child(1) > span > span")))


In [38]:
# driver = set_up_driver()
# medicine_links = get_medicine_links(driver, "https://www.pharmacity.vn/thuoc-khang-di-ung")
# print(f"Số liên kết thuốc thu thập được: {len(medicine_links)}")

# for link in medicine_links:
#     print(link)

In [39]:
if __name__ == "__main__":
    print("Bắt đầu thu thập liên kết thuốc từ Pharmacity")
    print("=" * 50)
    df = crawl_pharmacity_medicine_links()
    if isinstance(df, pd.DataFrame):
        print(f"Tổng số dòng: {len(df)}")
        display(df.head())
    print("\n" + "=" * 50)
    print("Hoàn tất thu thập liên kết thuốc từ Pharmacity")

Bắt đầu thu thập liên kết thuốc từ Pharmacity
Đang truy cập vào trang chủ Pharmacity...
Đang truy cập vào trang danh mục thuốc...
[1/4] Menu: Thuốc không kê đơn
  [1/16] Category: Thuốc ngừa thai
    -> Links: 5
      https://www.pharmacity.vn/vien-nen-postinor-1-hop-1-vien---ngua-thai-khan-cap.html
      https://www.pharmacity.vn/cerciorat-hop-1-vi-x-1-vien.html
      https://www.pharmacity.vn/bocinor-tab-15mg-hop-1-vi-x-1-vien.html
      https://www.pharmacity.vn/avalo-day-hop-28-vien.html
      https://www.pharmacity.vn/newlevo.html
  [2/16] Category: Thuốc kháng dị ứng
    -> Links: 107
      https://www.pharmacity.vn/dung-dich-uong-dieu-tri-viem-mui-di-ung-va-may-day-zyrtec-60ml.html
      https://www.pharmacity.vn/siro-dieu-tri-mat-ngu-va-cac-trieu-chung-di-ung-theralene-90ml.html
      https://www.pharmacity.vn/thuoc-dieu-tri-viem-mui-di-ung-va-cac-truong-hop-di-ung-ngoai-da-allerfar-4mg-10-vi-x-20-vien-hop.html
      https://www.pharmacity.vn/thuoc-dieu-tri-trieu-chung-viem-mui

,links,drug_type,category_link,category_name
0,https://www.pharmacity.vn/vien-nen-postinor-1-...,Thuốc không kê đơn,https://www.pharmacity.vn/thuoc-ngua-thai,Thuốc ngừa thai
1,https://www.pharmacity.vn/cerciorat-hop-1-vi-x...,Thuốc không kê đơn,https://www.pharmacity.vn/thuoc-ngua-thai,Thuốc ngừa thai
2,https://www.pharmacity.vn/bocinor-tab-15mg-hop...,Thuốc không kê đơn,https://www.pharmacity.vn/thuoc-ngua-thai,Thuốc ngừa thai
3,https://www.pharmacity.vn/avalo-day-hop-28-vie...,Thuốc không kê đơn,https://www.pharmacity.vn/thuoc-ngua-thai,Thuốc ngừa thai
4,https://www.pharmacity.vn/newlevo.html,Thuốc không kê đơn,https://www.pharmacity.vn/thuoc-ngua-thai,Thuốc ngừa thai



Hoàn tất thu thập liên kết thuốc từ Pharmacity


In [40]:
df.to_csv("pharmacity_medicine_links.csv")